# FinBERT sentiment for the news headlines

Scores every headline in the attached `hft-ml-data` dataset with ProsusAI/finbert, a
finance sentiment model trained in 2019 on older text, so it carries no knowledge of
later events. Needs a GPU and internet (to download the model). Writes
`finbert-scores.parquet` (id, positive, negative, neutral) and the model revision used.

In [ ]:
import json, zipfile
from pathlib import Path

import pyarrow as pa
import pyarrow.parquet as pq
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

inputs, work = Path("/kaggle/input"), Path("/kaggle/working")
news_dir = next(iter(inputs.rglob("news")), None)
if news_dir is None or not any(news_dir.glob("*.parquet")):
    for archive in inputs.rglob("*.zip"):
        with zipfile.ZipFile(archive) as z:
            z.extractall(work / "unpacked")
    news_dir = next((work / "unpacked").rglob("news"))
tables = [pq.read_table(p, columns=["id", "headline"]) for p in sorted(news_dir.glob("*.parquet"))]
news = pa.concat_tables(tables)
print(len(news), "headlines from", news_dir)

In [ ]:
MODEL = "ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL).eval().to("cuda")
revision = getattr(model.config, "_commit_hash", None)
labels = [model.config.id2label[i] for i in range(model.config.num_labels)]
print("labels", labels, "revision", revision)
headlines = news.column("headline").to_pylist()
probabilities = []
with torch.no_grad():
    for start in range(0, len(headlines), 512):
        batch = tokenizer(headlines[start:start + 512], padding=True, truncation=True, max_length=64, return_tensors="pt").to("cuda")
        probabilities.append(torch.softmax(model(**batch).logits.float(), dim=-1).cpu())
probabilities = torch.cat(probabilities).numpy()
columns = {"id": news.column("id")}
for i, name in enumerate(labels):
    columns[name] = pa.array(probabilities[:, i])
pq.write_table(pa.table(columns), work / "finbert-scores.parquet")
(work / "finbert-model.json").write_text(json.dumps({"model": MODEL, "revision": revision, "labels": labels, "rows": len(headlines)}))
print("done", probabilities.mean(axis=0))